# TrOCR run 2: consistent start token, attention LoRA, best CER
Pinned code, model and data; failure stops the notebook. Save outputs before session expiry.

Select GPU T4. CUDA computation is checked before downloading data/models.


In [ ]:
import subprocess, sys, os
from pathlib import Path
CODE_REVISION = "3b0afae8a511481e7593105dcb70aaf8be397a53"
BASE_REVISION = "93450be3f1ed40a930690d951ef3932687cc1892"
DATA_REVISION = "773832ea94643b630f63e8c2aa2002c634a7ade5"
repo = Path('/kaggle/working/OCR_engine')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/PiotrStyla/OCR_engine.git',str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','origin',CODE_REVISION],check=True)
subprocess.run(['git','-C',str(repo),'checkout','--detach',CODE_REVISION],check=True)
os.chdir(repo)
sys.path.insert(0,str(repo))
subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=True)
subprocess.run([sys.executable,'-m','pip','install','transformers==4.57.6','peft==0.19.1','jiwer','pillow','accelerate'],check=True)


In [ ]:
import torch
assert torch.cuda.is_available(), "GPU is required for the full evaluation; select a Kaggle GPU accelerator."
print('GPU:', torch.cuda.get_device_name(0), 'torch:', torch.__version__, 'architectures:', torch.cuda.get_arch_list())
# is_available() detects the device but does not prove wheel/kernel compatibility.
# Select GPU T4 in Kaggle (API: --accelerator NvidiaTeslaT4).
try:
    probe = torch.ones((2, 2), device='cuda')
    assert (probe @ probe).sum().item() == 8.0
    torch.cuda.synchronize()
    del probe
except Exception as exc:
    raise RuntimeError('GPU cannot execute this PyTorch build. Select Kaggle T4; P100 sm_60 is unsupported by the observed cu128 wheel.') from exc
print('CUDA_PREFLIGHT_OK', flush=True)
from huggingface_hub import snapshot_download
from training.protocol import pair_manifest
# Authenticate to avoid the shared Kaggle IP's anonymous HF rate limit.
from kaggle_secrets import UserSecretsClient
try:
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    raise RuntimeError('Enable the HF_TOKEN secret for this notebook in Kaggle Add-ons > Secrets.') from None
if not hf_token:
    raise RuntimeError('HF_TOKEN secret is empty')
os.environ['HF_TOKEN'] = hf_token
print('HF authentication configured (secret not displayed)', flush=True)
data_root = snapshot_download('PiotrSty/ocr-pl-lines',repo_type='dataset',revision=DATA_REVISION, token=hf_token, max_workers=2)
print('train:',len(pair_manifest(f'{data_root}/train')),'val:',len(pair_manifest(f'{data_root}/val')))


In [ ]:
# Run after inspecting first-run reevaluation. Start fresh from the pinned base.
output = '/kaggle/working/trocr-pl-run2'
subprocess.run([sys.executable,'-m','training.train_trocr_pl',
    '--train-dir',f'{data_root}/train','--val-dir',f'{data_root}/val',
    '--revision',BASE_REVISION,'--output',output,
    '--epochs','3','--batch-size','8','--no-4bit'],check=True)
print(Path(output,'selection.json').read_text())
print(Path(output,'best_metrics.json').read_text())
# No upload_folder: preserve the first model; review CER before any promotion.
